# BirdCLEF 2026 — Inference Notebook

**Model:** EfficientNet-B0 + Attention Pooling (SED)  |  **Classes:** 234  |  **Metric:** c-macro ROC AUC

> **Setup:** Add competition data (`birdclef-2026`) and your model weights via *Add Data* before running.  
> Internet: **OFF** · Accelerator: **None (CPU)**

In [ ]:
import sys
from pathlib import Path
import torch, torchaudio, timm

print(f"Python     : {sys.version.split()[0]}")
print(f"PyTorch    : {torch.__version__}")
print(f"torchaudio : {torchaudio.__version__}")
print(f"timm       : {timm.__version__}")

In [ ]:
# ── Paths ────────────────────────────────────────────────────────────────────
KAGGLE_INPUT = Path("/kaggle/input")

def _find(name: str) -> Path:
    matches = list(KAGGLE_INPUT.rglob(name))
    if not matches:
        raise FileNotFoundError(f"{name} not found under {KAGGLE_INPUT}")
    return matches[0]

TAXONOMY_PATH        = _find("taxonomy.csv")
COMPETITION_INPUT    = TAXONOMY_PATH.parent
TEST_SOUNDSCAPES_DIR = COMPETITION_INPUT / "test_soundscapes"
MODEL_PATH           = _find("best_model.pth")
SUBMISSION_PATH      = Path("/kaggle/working/submission.csv")

# ── Audio parameters (must match training) ───────────────────────────────────
SAMPLE_RATE    = 32_000
CHUNK_DURATION = 5
CHUNK_LENGTH   = SAMPLE_RATE * CHUNK_DURATION
N_MELS         = 128
N_FFT          = 1024
HOP_LENGTH     = 512
F_MIN          = 50
F_MAX          = 14_000
BATCH_SIZE     = 12

print(f"taxonomy      : {TAXONOMY_PATH}")
print(f"test_sounds   : {TEST_SOUNDSCAPES_DIR}")
print(f"model weights : {MODEL_PATH}")

In [ ]:
import torch
import torch.nn as nn
import torchaudio
import timm


class AttentionPooling(nn.Module):
    """Attention-weighted temporal aggregation over the time axis."""

    def __init__(self, in_features: int, hidden_dim: int = 512):
        super().__init__()
        self.attention = nn.Sequential(
            nn.Linear(in_features, hidden_dim),
            nn.Tanh(),
            nn.Linear(hidden_dim, 1),
            nn.Softmax(dim=1),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: (B, T, C)  →  out: (B, C)
        return torch.sum(x * self.attention(x), dim=1)


class SEDModel(nn.Module):
    """
    Sound Event Detection model for BirdCLEF 2026.
    Pipeline: waveform → log-Mel spectrogram → EfficientNet-B0
              → freq mean-pool → AttentionPool → linear classifier
    """

    def __init__(self, num_classes: int = 234, pretrained: bool = False):
        super().__init__()
        self.mel_spectrogram = torchaudio.transforms.MelSpectrogram(
            sample_rate=SAMPLE_RATE, n_fft=N_FFT, hop_length=HOP_LENGTH,
            n_mels=N_MELS, f_min=F_MIN, f_max=F_MAX,
        )
        self.amplitude_to_db = torchaudio.transforms.AmplitudeToDB()
        self.backbone = timm.create_model(
            "efficientnet_b0", pretrained=pretrained, in_chans=1
        )
        in_features = self.backbone.num_features
        self.backbone.classifier  = nn.Identity()
        self.backbone.global_pool = nn.Identity()
        self.pool       = AttentionPooling(in_features)
        self.classifier = nn.Linear(in_features, num_classes)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.amplitude_to_db(self.mel_spectrogram(x))  # (B, 1, n_mels, T)
        x = (x - x.mean()) / (x.std() + 1e-6)
        x = self.backbone.forward_features(x)              # (B, C, freq, T')
        x = x.mean(dim=2).transpose(1, 2)                  # (B, T', C)
        return self.classifier(self.pool(x))                # (B, num_classes)

In [ ]:
import pandas as pd

classes     = sorted(pd.read_csv(TAXONOMY_PATH)["primary_label"].unique())
num_classes = len(classes)

device = torch.device("cpu")
model  = SEDModel(num_classes=num_classes, pretrained=False).to(device)
model.load_state_dict(torch.load(MODEL_PATH, map_location=device, weights_only=True))
model.eval()

print(f"classes    : {num_classes}")
print(f"parameters : {sum(p.numel() for p in model.parameters()):,}")
print(f"weights    : {MODEL_PATH}")

In [ ]:
import soundfile as sf
import numpy as np
from tqdm.notebook import tqdm


def load_waveform(path: Path):
    """Load an .ogg file and return a mono 32 kHz waveform tensor."""
    waveform_np, sr = sf.read(path, dtype="float32", always_2d=True)
    waveform = torch.from_numpy(waveform_np).T              # (channels, frames)
    if sr != SAMPLE_RATE:
        waveform = torchaudio.transforms.Resample(sr, SAMPLE_RATE)(waveform)
    if waveform.shape[0] > 1:
        waveform = waveform.mean(dim=0, keepdim=True)       # stereo → mono
    return waveform


def predict(waveform: torch.Tensor) -> np.ndarray:
    """Run batched inference on a waveform; return sigmoid probabilities (T, C)."""
    duration   = waveform.shape[1] / SAMPLE_RATE
    n_chunks   = max(1, int(duration // CHUNK_DURATION))
    chunks     = []
    for i in range(n_chunks):
        chunk = waveform[:, i * CHUNK_LENGTH : (i + 1) * CHUNK_LENGTH]
        if chunk.shape[1] < CHUNK_LENGTH:
            chunk = torch.nn.functional.pad(chunk, (0, CHUNK_LENGTH - chunk.shape[1]))
        chunks.append(chunk)

    probs = []
    for b in range(0, len(chunks), BATCH_SIZE):
        batch = torch.stack(chunks[b : b + BATCH_SIZE]).to(device)
        probs.append(torch.sigmoid(model(batch)).cpu().numpy())
    return np.concatenate(probs, axis=0)                    # (n_chunks, num_classes)


# ── Resolve test soundscapes ──────────────────────────────────────────────────
test_files = sorted(TEST_SOUNDSCAPES_DIR.glob("*.ogg"))

# During Kaggle development the test audio is hidden; derive stems from
# sample_submission.csv so the notebook can still produce valid output.
if not test_files:
    _sample = pd.read_csv(COMPETITION_INPUT / "sample_submission.csv")
    stems   = sorted({"_".join(r.split("_")[:-1]) for r in _sample["row_id"]})
    source  = "sample_submission"
else:
    stems  = [f.stem for f in test_files]
    source = f"{len(test_files)} soundscapes"

print(f"source : {source}")

# ── Inference loop ────────────────────────────────────────────────────────────
rows = []

with torch.no_grad():
    for i, item in enumerate(tqdm(test_files if test_files else stems, desc="inference")):
        stem = item.stem if isinstance(item, Path) else item

        if test_files:
            try:
                waveform = load_waveform(item)
            except Exception as exc:
                print(f"  warning: {item.name} — {exc}")
                waveform = torch.zeros((1, SAMPLE_RATE * 60))
        else:
            waveform = torch.zeros((1, SAMPLE_RATE * 60))

        probs      = predict(waveform)
        n_chunks   = probs.shape[0]
        end_times  = [(j + 1) * CHUNK_DURATION for j in range(n_chunks)]

        for j, end_t in enumerate(end_times):
            row = {"row_id": f"{stem}_{end_t}"}
            row.update({cls: float(probs[j, k]) for k, cls in enumerate(classes)})
            rows.append(row)

print(f"rows generated : {len(rows)}")

In [ ]:
submission = pd.DataFrame(rows)[["row_id"] + classes]
submission.to_csv(SUBMISSION_PATH, index=False)

prob_cols = submission.iloc[:, 1:]
assert submission.shape[1] == 1 + num_classes, "unexpected column count"
assert ((prob_cols >= 0) & (prob_cols <= 1)).all().all(), "probabilities out of [0, 1]"

print(f"shape  : {submission.shape}")
print(f"output : {SUBMISSION_PATH}")
display(submission.head(3))